## 1. Load data into volumes from source - https zip from mkurans website


In [0]:
import requests
import zipfile
import io

# Link to mkuran's data source
url = "https://mkuran.pl/gtfs/warsaw.zip"

# Destination location
volume_dest = "/Volumes/warsaw_gtfs/default/text_data/"

# Get the data 
res = requests.get(url)

# Convert response to a file like format that the zipfile library can read
zip_file = zipfile.ZipFile(io.BytesIO(res.content))

# Create a list of names of the text files in the zip file to compare if exists later
text_files_list = zip_file.namelist()

volume_files = []

# Create a list of file names existing in the volume
for file_info_object in dbutils.fs.ls(volume_dest):
    volume_files.append(file_info_object.name)

# Loop through the text files in the zip file, if the file exists in the volume
# replace with the new data, if not just extract it.
for text_file in text_files_list:
    if text_file in volume_files:
        dbutils.fs.rm(f"{volume_dest}{text_file}")
        zip_file.extract(text_file, volume_dest)
        print(f"File: {text_file} overwritten with new data")
    else:
        print(f"File: {text_file} does not exist in volume, has been created")

# 2. Load the text files into delta tables

In [0]:
%sql 
drop table warsaw_gtfs.bronze.trips


In [0]:
# Write the files into delta tables in the bronze schema
for file in volume_files:
    df = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{volume_dest}{file}")
    df.write.mode("overwrite").format("delta").saveAsTable(f"warsaw_gtfs.bronze.{file.split(".", 1)[0]}")
    print(f"File: {file} has been written to table warsaw_gtfs.bronze.{file.split(".", 1)[0]}")